In [2]:
# Block 0: Install libraries (run this only once per session if needed)
!pip install PyMuPDF langdetect
import nltk
import os

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:  # Catch LookupError for missing resources
    # Download 'punkt' to a specific directory to avoid permission issues
    nltk.download('punkt', download_dir=os.path.expanduser("~/.nltk_data"))
    # Add the download directory to NLTK's data path
    nltk.data.path.append(os.path.expanduser("~/.nltk_data"))
# Download the required resource for NLTK
nltk.download('punkt_tab') # This line will download the necessary data

print("Libraries ready.")

[nltk_data] Downloading package punkt to /root/.nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Libraries ready.


In [3]:
# Block 1A: Upload PDF and extract text
import fitz  # PyMuPDF
from google.colab import files
import io

uploaded = files.upload()

text_content = ""
if uploaded:
    # Assuming only one file is uploaded
    file_name = next(iter(uploaded))
    file_content = uploaded[file_name]

    try:
        # Open the PDF from bytes
        doc = fitz.open(stream=file_content, filetype="pdf")
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            text_content += page.get_text()
        doc.close()
        print(f"Successfully extracted text from '{file_name}'.")
        print("\nFirst 500 characters of the extracted text:")
        print(text_content[:500] + "...")
    except Exception as e:
        print(f"Error processing PDF: {e}")
        text_content = None # Ensure text_content is None if error
else:
    print("No file uploaded.")
    text_content = None # Ensure text_content is None if no file

# Store the raw text for later comparison if needed
raw_text = text_content

Saving Convocatoria_de_propuesta_Contramaquinas_1746217385029_0.pdf to Convocatoria_de_propuesta_Contramaquinas_1746217385029_0.pdf
Successfully extracted text from 'Convocatoria_de_propuesta_Contramaquinas_1746217385029_0.pdf'.

First 500 characters of the extracted text:
CONVOCATORIA DE PROPUESTA 
__________________________________________________ 
Contramáquinas y archivos desobedientes: Arte-factos de cuidado frente a la 
necromáquina 
Editado por  
Alina Peña Iguarán, Patricio Azócar Donoso, Ana Cornide, Tatiana Navallo 
Este volumen colectivo surge en un tiempo donde las violencias contemporáneas han dejado 
de ser la excepción para convertirse en el tejido mismo de la organización social, afectiva y 
política del mundo. Como advierten Judith Butler y Athena...


In [4]:
# Block 2: Identify language
from langdetect import detect, LangDetectException

if text_content:
    try:
        language = detect(text_content)
        print(f"Detected language: {language}")
        if language == 'es':
            print("The text is confirmed to be in Spanish.")
        else:
            print(f"Warning: The detected language is '{language}', not Spanish. Proceeding, but results might be inaccurate.")
    except LangDetectException:
        print("Could not detect language (text might be too short or ambiguous). Assuming Spanish for now.")
        language = 'es' # Default to Spanish if detection fails
else:
    print("No text content available to detect language.")
    language = None # Or some other indicator that processing should stop

Detected language: es
The text is confirmed to be in Spanish.


In [9]:
# Block 3: Erase Spanish articles and prepositions

# Define Spanish articles and common prepositions
# (This list can be expanded for more thoroughness)
spanish_articles = [
    "el", "la", "los", "las", "un", "una", "unos", "unas", "lo" # added "lo" (neuter article)
]
spanish_prepositions = [
    "a", "ante", "bajo", "cabe", "con", "contra", "de", "desde", "durante",
    "en", "entre", "hacia", "hasta", "mediante", "para", "por", "que", "según",
    "sin", "so", "sobre", "tras", "versus", "vía", "al", "o", "y", "le", "su"
]
stopwords_es = set(spanish_articles + spanish_prepositions) # Use a set for faster lookups

text_without_stopwords = ""
if text_content and (language == 'es' or language is None): # Proceed if Spanish or if lang detection failed
    # Simple tokenization by splitting by space, after converting to lowercase
    # More robust tokenization could use nltk.word_tokenize, but for stopword removal,
    # this is often sufficient and avoids issues with punctuation attached to words
    # if we clean punctuation before this step.

    # For better stopword removal, let's clean punctuation first, then tokenize
    import re
    # Remove punctuation, keeping only alphanumeric characters and spaces
    cleaned_for_stopwords = re.sub(r'[^\w\s]', '', text_content.lower())
    words = cleaned_for_stopwords.split()

    # Filter out stopwords
    filtered_words = [word for word in words if word not in stopwords_es]

    text_without_stopwords = " ".join(filtered_words)

    print("Text after removing Spanish articles and prepositions:")
    print(text_without_stopwords[:500] + "...")
elif not text_content:
    print("No text content to process.")
else:
    print(f"Skipping stopword removal as language is not Spanish (detected: {language}).")
    text_without_stopwords = text_content # Pass original text if not Spanish

Text after removing Spanish articles and prepositions:
convocatoria propuesta __________________________________________________ contramáquinas archivos desobedientes artefactos cuidado frente necromáquina editado alina peña iguarán patricio azócar donoso ana cornide tatiana navallo este volumen colectivo surge tiempo donde violencias contemporáneas han dejado ser excepción convertirse tejido mismo organización social afectiva política del mundo como advierten judith butler athena athanasiou 2022 capitalismo neoliberal no solo distribuye precariedad...


In [10]:
# Block 4: Normalize accented characters
text_normalized = text_without_stopwords

# Specific replacements as requested
replacements = {
    "á": "a", "é": "e", "í": "i", "ó": "o", "ú": "u", "ü": "u",
    "Á": "A", "É": "E", "Í": "I", "Ó": "O", "Ú": "U", "Ü": "U"
    # ñ is already ñ, so no change needed, but explicitly mentioned in requirement.
    # Ñ to ñ if we lowercase everything, or Ñ to N if we want to preserve case.
    # For simplicity and consistency with previous lowercasing, let's ensure ñ is handled.
    # If the text was already lowercased, uppercase replacements are not strictly needed
    # but good for robustness if the previous step didn't fully lowercase.
}

# The previous step likely lowercased everything.
# If not, uncomment this: text_normalized = text_normalized.lower()

for original, replacement in replacements.items():
    text_normalized = text_normalized.replace(original, replacement)

# Ensure ñ is preserved correctly (it should be already)
# text_normalized = text_normalized.replace("ñ", "ñ") # This line is redundant

print("Text after normalizing accents:")
print(text_normalized[:500] + "...")

Text after normalizing accents:
convocatoria propuesta __________________________________________________ contramaquinas archivos desobedientes artefactos cuidado frente necromaquina editado alina peña iguaran patricio azocar donoso ana cornide tatiana navallo este volumen colectivo surge tiempo donde violencias contemporaneas han dejado ser excepcion convertirse tejido mismo organizacion social afectiva politica del mundo como advierten judith butler athena athanasiou 2022 capitalismo neoliberal no solo distribuye precariedad...


In [11]:
# Block 5: Identify bigrams
from nltk.tokenize import word_tokenize
from nltk.util import bigrams as nltk_bigrams # Alias to avoid conflict if 'bigrams' var used
import re

bigram_list = []
if text_normalized:
    # Tokenize the text. word_tokenize is good at handling punctuation.
    # Convert to lowercase again to ensure consistency for bigrams
    tokens = word_tokenize(text_normalized.lower())

    # Filter out tokens that are purely punctuation or non-alphanumeric
    # This helps in getting meaningful word bigrams
    alphanumeric_tokens = [token for token in tokens if token.isalnum()]

    # Generate bigrams
    # nltk_bigrams returns an iterator, convert to list
    bigram_list = list(nltk_bigrams(alphanumeric_tokens))

    print(f"Found {len(bigram_list)} bigrams.")
    print("First 10 bigrams:")
    for i, bigram in enumerate(bigram_list[:10]):
        print(f"{i+1}. ('{bigram[0]}', '{bigram[1]}')")
else:
    print("No text to process for bigrams.")

Found 447 bigrams.
First 10 bigrams:
1. ('convocatoria', 'propuesta')
2. ('propuesta', 'contramaquinas')
3. ('contramaquinas', 'archivos')
4. ('archivos', 'desobedientes')
5. ('desobedientes', 'artefactos')
6. ('artefactos', 'cuidado')
7. ('cuidado', 'frente')
8. ('frente', 'necromaquina')
9. ('necromaquina', 'editado')
10. ('editado', 'alina')


In [12]:
# Block 6: Generate a .csv file with the bigrams
import pandas as pd

if bigram_list:
    # Create a DataFrame from the list of bigrams
    df_bigrams = pd.DataFrame(bigram_list, columns=['Word1', 'Word2'])

    # Define CSV file name
    csv_file_name = "spanish_bigrams.csv"

    # Save to CSV
    df_bigrams.to_csv(csv_file_name, index=False, encoding='utf-8')

    print(f"Successfully generated '{csv_file_name}' with {len(df_bigrams)} bigrams.")
    print("You can download it from the Colab file browser (folder icon on the left).")

    # Optional: provide a direct download link (works in most Colab environments)
    try:
        files.download(csv_file_name)
        print(f"Download initiated for '{csv_file_name}'.")
    except Exception as e:
        print(f"Could not initiate direct download: {e}. Please use the file browser.")

    # Display first few rows of the DataFrame
    print("\nPreview of the CSV content (first 5 rows):")
    print(df_bigrams.head())

elif text_normalized and not bigram_list: # Text existed but no bigrams (e.g., less than 2 words)
    print("No bigrams were found (text might be too short after processing). CSV not generated.")
else:
    print("No bigrams to save to CSV.")

Successfully generated 'spanish_bigrams.csv' with 447 bigrams.
You can download it from the Colab file browser (folder icon on the left).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download initiated for 'spanish_bigrams.csv'.

Preview of the CSV content (first 5 rows):
            Word1           Word2
0    convocatoria       propuesta
1       propuesta  contramaquinas
2  contramaquinas        archivos
3        archivos   desobedientes
4   desobedientes      artefactos
